In [36]:
import pickle
import re 
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_precompute import CommutingGateRouterPrecompute
from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz import CommutingGateRouterPrecomputeRzz

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
ess = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = ess._num_vertices
max_layers = 0

Keeping constraints at times: [0]


In [4]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecompute(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [5]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

Max layers needed to apply swap decompose: 0
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Gates we cannot directly implement: 10
[(2, 5), (0, 1, 5), (0, 1, 4), (1, 2, 4, 5), (1, 3, 4), (1, 5), (0, 1, 3, 4), (0, 2, 5), (0, 4, 5), (0, 3, 4, 5)]
Transpiling un-implemented gates


In [6]:
tqc.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                           ┌───┐                                                                                                                 ┌───┐                                                                                                                                                                            ┌───┐                                ┌───┐                                                                  ┌───┐┌───────────┐┌───┐                   ┌───┐┌───────────┐┌───┐                                                                                                                      ┌───┐┌───────────┐┌───┐┌───┐┌────────────┐┌───┐                                                                               
q_0 -> 0 ┤ Rz(8.875) ├───────■───────────────────────────────────────────────────────────────────┤ X ├──■────────■─────────────────────────────────────────────────────────────────────────────────────────────■────■──┤ X ├───────────────────────────■───────────────────────■──────────────────────────────────────────────────────────────────■───────────────────────────■─────────────────■───────┤ X ├──■─────────────────────■───────┤ X ├──────────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.625) ├┤ X ├───────────────────┤ X ├┤ Rz(1.125) ├┤ X ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(1.125) ├┤ X ├┤ X ├┤ Rz(-0.625) ├┤ X ├───────X───────────────────────────────────────────────────────────────────────
         └───────────┘     ┌─┴─┐                                                                 └─┬─┘  │      ┌─┴─┐                                                                                         ┌─┴─┐  │  └─┬─┘                         ┌─┴─┐                     │                                                                  │       ┌───┐┌───────────┐┌─┴─┐┌───────────┐┌─┴─┐┌───┐└─┬─┘  │                     │       └─┬─┘                                                             ┌───┐└─┬─┘└───────────┘└─┬─┘┌───┐              └─┬─┘└───────────┘└─┬─┘                                                                                                            ┌───┐     └─┬─┘└───────────┘└─┬─┘└─┬─┘└────────────┘└─┬─┘┌───┐  │                                                                       
q_1 -> 1 ──────────────────┤ X ├───────■──────────────────────────────────────────────────■────────■────┼──────┤ X ├──────■───────────────────────────■─────────────────────────────────────────■─────────■──┤ X ├──┼────■────■───────────────────■──┤ X ├─────────────────────┼──────────────────────────────────────────────────────────────────┼───────┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ X ├──■────┼─────────────────────┼─────────■───────────────────────────────────────────────────────────────┤ X ├──■─────────────────■──┤ X ├─X─────X────────┼─────────────────┼────────────────X───────────────────────────────────X─────────────────────────────────────────────────────────┤ X ├───────┼─────────────────┼────■──────────────────■──┤ X ├──┼───────────────────────────────────────────────────────────────────────
                           └───┘       │                                                  │           ┌─┴─┐┌───┴───┴───┐  │                           │                                         │         │  └───┘┌─┴─┐       │                   │  └───┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌───┐┌────────────┐                        ┌─┴─┐┌───┐└─┬─┘└───────────┘└───┘└───────────┘└───┘└─┬─┘     ┌─┴─┐┌───────────┐    ┌─┴─┐          ┌───┐                        ┌───┐                           └─┬─┘                       └─┬─┘ │     │ ┌───┐  │                 │  ┌───┐         │      ┌───┐┌───────────┐┌───┐      │           ┌───┐┌───────────┐┌───┐                       └─┬─┘┌───┐  │                 │  ┌───┐                   └─┬─┘  │

In [7]:
print_circuit_info(tqc, 'TQC')

TQC has 6 qubits
TQC has 110 non-local gates and 87 non-local depth
TQC contains ['cx', 'rz', 'swap'] gates.


In [8]:
with open('/lustre/scratch127/qpg/jc59/circuit_depths/results.precompute.120.pkl', 'rb') as f:
    res = pickle.load(f)
for k, v in res.items():
    print(k, v['default'][::-1], v['rz'][0:3]) #, v['rzz'])

test_N4_W6 (4039, 4882) [1712, 4109, 52]
test_N2_W2 (95, 106) [70, 90, 2]
trivial (320, 369) [171, 240, 8]
test_N3_W4 (511, 604) [229, 399, 7]
test_N4_W5 (3127, 3729) [1486, 3092, 23]


In [9]:
qc = res['test_N2_W2']['rz'][4]
qc.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                                                                                                                                                                                                    ┌───┐                        ┌───┐                                                                                                               ┌───┐┌────────────┐                        ┌───┐                                                                                                                                             
q_0 -> 0 ┤ Rz(8.875) ├───────■──────────────────────────────────────────────────────────────────────────────────■─────────────────────────────────────────────■─────────────────────────────■──────────────────────────────────────────────────────────────────────────■────────────────────────────────────────────────────────────────────────────────────■─────────────────────────────────────────────┤ X ├──■──────────────────■──┤ X ├──X───────────────────────────■───────────────────────────■────────────────────────────────────────────────────┤ X ├┤ Rz(-0.625) ├──■──────────────────■──┤ X ├───────────X─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         └───────────┘     ┌─┴─┐                                                                                │                                             │                           ┌─┴─┐                                                                        │                                                                                    │                                             └─┬─┘  │                  │  └─┬─┘  │                           │                           │                                                    └─┬─┘└────────────┘┌─┴─┐┌────────────┐┌─┴─┐└─┬─┘           │                                                                                                                                 
q_1 -> 1 ──────────────────┤ X ├───────■──────────────────────────■──────────────────────────■──────────────────┼─────────────────────────────────────────────┼────────────────────────■──┤ X ├──■─────────────────────────────────────────────────■───────────────────┼────────────────────────────────────────────────────────────────────────────────────┼────■─────────────────────────────────────■────■────┼──────────────────┼────■────X───────────────────────────┼────■──────────────────────┼───────────────────────────────■──────────────────────┼────────────────┤ X ├┤ Rz(-0.625) ├┤ X ├──┼────■────────┼───────────────────────────────────────────────────■───────────────────────────────────────────────────X─────────────────────────
                           └───┘       │                          │                          │  ┌───┐         ┌─┴─┐    ┌───────────┐                        ┌─┴─┐     ┌───┐            │  └───┘  │                                                 │                 ┌─┴─┐                                                                                ┌─┴─┐  │  ┌───┐                       ┌───┐  │       ┌─┴─┐┌────────────┐┌─┴─┐                                 ┌─┴─┐  │  ┌────────────┐    ┌─┴─┐                             │           ┌───┐      │                └───┘└────────────┘└───┘  │    │  ┌───┐ │ ┌───┐┌────────────┐                        ┌───┐  │  ┌───┐┌────────────┐                        ┌───┐ │                         
q_2 -> 2 ──────■───────────────────────┼──────────────────────────┼────────■─────────────────┼──┤ X ├──■──────┤ X ├────┤ Rz(0.625) ├──■──────────────────■──┤ X ├──■──┤ X ├──■─────────┼─────────┼─────────────────────────────────────────────────┼──────────────■──┤ X ├──■────────────────────────────■────■──────────────────────────────

In [10]:
pm_rzz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecomputeRzz(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [11]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc_rzz = pm_rzz.run(qc)

Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 9
[(0, 1, 5), (1, 4), (0, 1, 4), (1, 2, 4, 5), (1, 3, 4), (1, 5), (0, 1, 3, 4), (0, 2, 5), (0, 4, 5)]
Transpiling un-implemented gates


In [12]:
tqc_rzz.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                                                                          ┌───┐                      ┌───┐            ┌───┐     ┌───┐                                                                                                                                                                                                                      ┌───┐┌───────────┐┌───┐                                                               ┌───┐                                                    
q_0 -> 0 ┤ Rz(8.875) ├───────■─────────────────────────────────────────────────■──────────────────■──────────────────────────────────────────────────■────────■────────────────────────────────────────────────────────────────────────────────────────■─────────────────────■──┤ X ├──────■────────────■──┤ X ├─■──────────┤ X ├──■──┤ X ├──■─────────────────────────────────────────────■────────────X──────────────────────────────────────────────────────────────X───■─────────────────■───X────────────────────────────────────────────X──────────────────┤ X ├┤ Rz(1.125) ├┤ X ├─X──────────────────────────■───────────────────────────■──────┤ X ├────────────────────────────────────────────────────
         └───────────┘     ┌─┴─┐                                               │                  │                                                ┌─┴─┐      │                                                                                        │                     │  └─┬─┘      │            │  └─┬─┘ │          └─┬─┘  │  └─┬─┘  │                                             │            │  ┌───┐┌───────────┐┌───┐                                     │ ┌─┴─┐┌───────────┐┌─┴─┐ │                                            │                  └─┬─┘└───────────┘└─┬─┘ │                          │                           │      └─┬─┘                       ┌───┐┌────────────┐┌───┐     
q_1 -> 1 ──────────────────┤ X ├──────────■────────────────────────────────────┼──────────────────┼───■─────────────────────────────■──────────────┤ X ├──────┼────■─────────────■──────────────────────────────────■──────────────────────────────────┼─────────────────────┼────■────────┼────────────┼────■───┼────────────■────┼────■────┼─────────────────────────────────────────────┼────────X───X──┤ X ├┤ Rz(0.625) ├┤ X ├─────────────────────────────────────┼─┤ X ├┤ Rz(1.125) ├┤ X ├─┼────────────────────────────────────────────┼────────────────X───┼─────────────────┼───┼──────────────────────────┼───────────────────────────┼────────■──────────────X──────────┤ X ├┤ Rz(-0.625) ├┤ X ├─────
                           └───┘          │                              ┌───┐ │ZZ(-0.625)      ┌─┴─┐ │                             │              └───┘    ┌─┴─┐  │  ┌───┐      │                                  │                                ┌─┴─┐                 ┌─┴─┐           │ZZ(0.625) ┌─┴─┐      │ZZ(0.625)      ┌─┴─┐     ┌─┴─┐                       ┌───┐             ┌─┴─┐┌───┐ │      └─┬─┘└───────────┘└─┬─┘                               ┌───┐ │ └───┘└───────────┘└───┘ │ ┌───┐             ┌───┐┌───────────┐┌───┐  │          ┌───┐ │   │                 │   │                        ┌─┴─┐                       ┌─┴─┐                     │          └─┬─┘└────────────┘└─┬─┘     
q_2 -> 2 ───────────────■─────────────────┼───────■──────────────────────┤ X ├─■─────────────■──┤ X ├─┼─────────────────────────■───┼───────────────────────┤ X ├──┼──┤ X ├──■───┼──────────────────────────────────┼───────────■─────────────■──────┤ X ├─────■───────────┤ X ├──■────■───■──────────┤ X ├──■───■────────────■──┤ X ├─────┤ X ├──■─────────────────■──┤ X ├─■───────────┤ X ├┤ X ├─┼────────┼─────────────────┼────────X───■─────────────────■──┤ X ├─X─────────────────────────X─┤ X ├─────────────┤ X ├┤ Rz(0.625) ├┤ X ├──X──────

In [13]:
print_circuit_info(tqc_rzz, 'TQC Rzz')

TQC Rzz has 6 qubits
TQC Rzz has 100 non-local gates and 80 non-local depth
TQC Rzz contains ['cx', 'rzz', 'swap', 'rz'] gates.


In [14]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap"]
method = 'unitary'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=basis_gates
)

config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

Num qubits in backend: 6


In [15]:
swaps = []
for instruction in tqc.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc.swap(*swap)
tqc.save_unitary()


swaps = []
for instruction in tqc_rzz.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc_rzz.swap(*swap)
tqc_rzz.save_unitary()

In [39]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
t_default = transpile(default, backend, optimization_level=3)
t_default.save_unitary()
res_default = backend.run(t_default.assign_parameters([1])).result()
u_def = np.asarray(res_default.get_unitary())

In [40]:
res_rz = backend.run(tqc).result()
u_rz = np.asarray(res_rz.get_unitary())

In [41]:
res_rzz = backend.run(tqc_rzz).result()
u_rzz = np.asarray(res_rzz.get_unitary())

In [43]:
np.nonzero((u_rzz - u_def) > 1e-5)

(array([], dtype=int64), array([], dtype=int64))